# 01 — Data Cleaning

Validate, clean, and prepare all datasets for downstream analysis.
Outputs cleaned parquet files to `data/processed/` with explicit schemas.

In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.metrics import EDGE_LIST_SCHEMA, FILE_METRICS_SCHEMA, TARGET_METRICS_SCHEMA

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

cfg = Config.from_yaml('../config.yaml')
file_df   = pd.read_parquet(cfg.processed_data_dir / 'file_metrics.parquet')
target_df = pd.read_parquet(cfg.processed_data_dir / 'target_metrics.parquet')
edge_df   = pd.read_parquet(cfg.processed_data_dir / 'edge_list.parquet')
G = load_graph(cfg.processed_data_dir / 'edge_list.parquet')
attach_metrics(G, target_df)

print(f'file_df   : {file_df.shape}')
print(f'target_df : {target_df.shape}')
print(f'edge_df   : {edge_df.shape}')
print(f'Graph     : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 1. Missing Data Identification

In [ ]:
# --- Null heatmap: file_metrics ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

msno.heatmap(file_df, ax=axes[0], fontsize=7)
axes[0].set_title('file_metrics nulls')

msno.heatmap(target_df, ax=axes[1], fontsize=7)
axes[1].set_title('target_metrics nulls')

msno.heatmap(edge_df, ax=axes[2], fontsize=7)
axes[2].set_title('edge_list nulls')

plt.tight_layout()
plt.show()

In [ ]:
# Non-compilable files: language is NaN (e.g. headers, resources, CMake scripts)
non_compilable = file_df[file_df['language'].isna()]
print(f'Non-compilable files (language=NaN): {len(non_compilable)}')
print(non_compilable[['source_file', 'cmake_target']].head(10))

# INTERFACE/UTILITY targets with no compile time
iface_targets = target_df[
    target_df['target_type'].isin(['INTERFACE_LIBRARY', 'UTILITY']) &
    target_df['compile_time_sum_ms'].isna()
]
print(f'\nINTERFACE/UTILITY targets with no compile time: {len(iface_targets)}')
print(iface_targets[['cmake_target', 'target_type']].head(10))

In [ ]:
# Generated files must have git_commit_count == 0
generated_with_history = file_df[
    file_df['is_generated'].eq(True) &
    file_df['git_commit_count'].gt(0)
]
assert len(generated_with_history) == 0, (
    f'{len(generated_with_history)} generated files have git history — '
    f'check is_generated flag or git history collector:\n{generated_with_history[["source_file", "git_commit_count"]]}'
)
print('Assertion passed: all generated files have git_commit_count == 0')

## 2. Outlier Detection

In [ ]:
# IQR outlier detection on compile_time_ms
ct = file_df['compile_time_ms'].dropna()
Q1, Q3 = ct.quantile(0.25), ct.quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outlier_mask = (file_df['compile_time_ms'] > upper_fence) | (file_df['compile_time_ms'] < lower_fence)
outliers = file_df[outlier_mask].copy()
print(f'Q1={Q1:.0f}ms  Q3={Q3:.0f}ms  IQR={IQR:.0f}ms')
print(f'Lower fence: {lower_fence:.0f}ms  Upper fence: {upper_fence:.0f}ms')
print(f'Outlier files: {len(outliers)} / {len(file_df)} ({100*len(outliers)/len(file_df):.1f}%)')
outliers[['source_file', 'cmake_target', 'compile_time_ms']].sort_values('compile_time_ms', ascending=False).head(15)

In [ ]:
# Box / strip plot with outliers highlighted
plot_df = file_df[['source_file', 'compile_time_ms']].dropna().copy()
plot_df['is_outlier'] = plot_df['compile_time_ms'].gt(upper_fence)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
axes[0].boxplot(plot_df['compile_time_ms'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0].set_ylabel('compile_time_ms')
axes[0].set_title('Compile Time Distribution (box)')
axes[0].set_xticks([])

# Strip plot
normal = plot_df[~plot_df['is_outlier']]
out = plot_df[plot_df['is_outlier']]
axes[1].scatter(np.zeros(len(normal)), normal['compile_time_ms'],
                alpha=0.3, s=8, color='steelblue', label='Normal')
axes[1].scatter(np.zeros(len(out)), out['compile_time_ms'],
                alpha=0.9, s=30, color='crimson', marker='^', label='Outlier')
axes[1].axhline(upper_fence, color='crimson', linestyle='--', linewidth=1, label=f'Upper fence ({upper_fence:.0f}ms)')
axes[1].set_ylabel('compile_time_ms')
axes[1].set_title('Compile Time Strip (outliers in red)')
axes[1].legend()
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

In [ ]:
# Clip outliers to IQR fence for time-based analyses
file_df['compile_time_ms_clipped'] = file_df['compile_time_ms'].clip(lower=max(0, lower_fence), upper=upper_fence)
print('Added compile_time_ms_clipped column (capped at IQR fences)')
print(file_df['compile_time_ms_clipped'].describe())

## 3. Path Alignment

In [ ]:
import os

# Canonicalise file paths using the source directory as the base
base_dir = str(cfg.source_dir)

def canonicalise(path: str) -> str:
    if os.path.isabs(path):
        return os.path.realpath(path)
    return os.path.realpath(os.path.join(base_dir, path))

file_df['source_file_canonical'] = file_df['source_file'].apply(canonicalise)
print('Canonicalised source_file paths')
print(file_df[['source_file', 'source_file_canonical']].head(5))

In [ ]:
# Cross-reference against files.json (CMake File API)
files_json_path = cfg.build_dir / '.cmake' / 'api' / 'v1' / 'reply' / 'files.json'
if files_json_path.exists():
    with open(files_json_path) as f:
        files_json = json.load(f)
    cmake_paths = set(
        os.path.realpath(p) for p in files_json.get('paths', {}).get('source', [])
        if p  # skip empty
    )
    parquet_paths = set(file_df['source_file_canonical'])

    in_parquet_not_cmake = parquet_paths - cmake_paths
    in_cmake_not_parquet = cmake_paths - parquet_paths
    print(f'Files in parquet but not in files.json : {len(in_parquet_not_cmake)}')
    print(f'Files in files.json but not in parquet : {len(in_cmake_not_parquet)}')
    if in_parquet_not_cmake:
        print('Sample (parquet only):', list(in_parquet_not_cmake)[:5])
    if in_cmake_not_parquet:
        print('Sample (cmake only):', list(in_cmake_not_parquet)[:5])
else:
    print(f'files.json not found at {files_json_path} — skipping cross-reference')

## 4. Target Validation

In [ ]:
# Targets in graph vs targets in target_df
graph_targets  = set(G.nodes())
metric_targets = set(target_df['cmake_target'])

phantom_targets  = graph_targets - metric_targets   # in graph, not built
unlisted_targets = metric_targets - graph_targets   # built, not in graph

print(f'Graph targets     : {len(graph_targets)}')
print(f'Metric targets    : {len(metric_targets)}')
print(f'Phantom targets   : {len(phantom_targets)}')
print(f'Unlisted targets  : {len(unlisted_targets)}')

if phantom_targets:
    print('\nPhantom (in graph, not in metrics):')
    for t in sorted(phantom_targets)[:20]:
        print(f'  {t}')

if unlisted_targets:
    print('\nUnlisted (in metrics, not in graph):')
    for t in sorted(unlisted_targets)[:20]:
        print(f'  {t}')

In [ ]:
# Alias targets: target_type == 'ALIAS' or name contains '::'
alias_targets = target_df[
    target_df['target_type'].eq('ALIAS') |
    target_df['cmake_target'].str.contains('::', na=False)
]
print(f'Alias targets: {len(alias_targets)}')
print(alias_targets[['cmake_target', 'target_type']].head(10))

## 5. Codegen Validation

In [ ]:
# Coverage check: codegen_inventory.json vs is_generated in file_df
codegen_inventory_path = cfg.raw_data_dir / 'codegen_inventory.json'
if codegen_inventory_path.exists():
    with open(codegen_inventory_path) as f:
        inventory = json.load(f)
    inventory_paths = set(os.path.realpath(p) for p in inventory.get('generated_files', []))
    parquet_generated = set(file_df.loc[file_df['is_generated'].eq(True), 'source_file_canonical'])

    in_inventory_not_parquet = inventory_paths - parquet_generated
    in_parquet_not_inventory = parquet_generated - inventory_paths

    print(f'Inventory generated files : {len(inventory_paths)}')
    print(f'Parquet is_generated=True : {len(parquet_generated)}')
    print(f'In inventory, not parquet : {len(in_inventory_not_parquet)}')
    print(f'In parquet, not inventory : {len(in_parquet_not_inventory)}')
else:
    print(f'codegen_inventory.json not found at {codegen_inventory_path} — using is_generated flag only')

# Verify generated files have no git history
gen_with_git = file_df[file_df['is_generated'].eq(True) & file_df['git_commit_count'].gt(0)]
print(f'\nGenerated files with git history (should be 0): {len(gen_with_git)}')
if len(gen_with_git) > 0:
    print('WARNING — these files are marked generated but have git commits:')
    print(gen_with_git[['source_file', 'git_commit_count']].head(10))

## 6. Contributor Data Validation

In [ ]:
contributors_path = cfg.raw_data_dir / 'contributors.csv'
contributors_df = pd.read_csv(contributors_path)
print(f'Contributors: {len(contributors_df)}')
print(contributors_df.dtypes)
contributors_df.head()

In [ ]:
# Detect aliases: contributors whose email local-part matches but domain differs
# e.g. alice@company.com  vs  alice@personal.io
contributors_df['email_local'] = contributors_df['contributor'].str.split('@').str[0].str.lower()

alias_groups = contributors_df.groupby('email_local').filter(lambda g: len(g) > 1)
if len(alias_groups) > 0:
    print(f'Potential email aliases ({len(alias_groups)} rows):')
    print(alias_groups[['contributor', 'email_local', 'total_commits']].sort_values('email_local'))
else:
    print('No email alias candidates found')

In [ ]:
# Flag bot accounts: emails containing 'bot', 'ci', 'jenkins', 'noreply', 'automation'
bot_pattern = re.compile(r'bot|\bci\b|jenkins|noreply|automation|dependabot|renovate', re.IGNORECASE)
bot_mask = contributors_df['contributor'].str.contains(bot_pattern, na=False)
bots = contributors_df[bot_mask].copy()
print(f'Bot/CI accounts flagged: {len(bots)}')
print(bots[['contributor', 'total_commits']].to_string())

In [ ]:
# Build alias mapping and merge duplicates
# Strategy: for each email_local group, keep the email with the highest total_commits as canonical
alias_map: dict[str, str] = {}  # original -> canonical

for local, group in contributors_df.groupby('email_local'):
    if len(group) > 1:
        canonical = group.loc[group['total_commits'].idxmax(), 'contributor']
        for email in group['contributor']:
            if email != canonical:
                alias_map[email] = canonical

print(f'Alias mappings: {len(alias_map)}')
for orig, canon in list(alias_map.items())[:10]:
    print(f'  {orig!r} -> {canon!r}')

# Apply alias mapping to contributors_df
contributors_df['contributor_canonical'] = contributors_df['contributor'].replace(alias_map)

# Remove bots from analysis
contributors_clean = contributors_df[~bot_mask].copy()
print(f'\nClean contributor count (no bots): {len(contributors_clean)}')

## 7. Type Casting and Normalisation

In [ ]:
# Dtype audit against PyArrow schemas
def audit_schema(df: pd.DataFrame, schema: pa.Schema, label: str) -> pd.DataFrame:
    """Return a DataFrame showing current vs expected dtype for each schema field."""
    rows = []
    for field in schema:
        col = field.name
        if col in df.columns:
            rows.append({
                'column': col,
                'current_dtype': str(df[col].dtype),
                'expected_pa': str(field.type),
                'null_count': int(df[col].isna().sum()),
            })
        else:
            rows.append({
                'column': col,
                'current_dtype': 'MISSING',
                'expected_pa': str(field.type),
                'null_count': None,
            })
    audit = pd.DataFrame(rows)
    print(f'\n=== {label} ===')
    missing = audit[audit['current_dtype'] == 'MISSING']
    if len(missing):
        print(f'MISSING columns ({len(missing)}):', missing['column'].tolist())
    return audit

file_audit   = audit_schema(file_df, FILE_METRICS_SCHEMA, 'file_metrics')
target_audit = audit_schema(target_df, TARGET_METRICS_SCHEMA, 'target_metrics')
edge_audit   = audit_schema(edge_df, EDGE_LIST_SCHEMA, 'edge_list')

print('\nfile_metrics dtype audit (first 15 rows):')
file_audit.head(15)

In [ ]:
# Normalise: ensure time columns are int64 (ms) and size columns are int64 (bytes)
time_cols_file = ['compile_time_ms', 'gcc_parse_time_ms', 'gcc_template_instantiation_ms',
                  'gcc_codegen_time_ms', 'gcc_optimization_time_ms', 'gcc_total_time_ms']
size_cols_file = ['source_size_bytes', 'preprocessed_bytes', 'object_size_bytes']

for col in time_cols_file:
    if col in file_df.columns:
        file_df[col] = pd.to_numeric(file_df[col], errors='coerce')

for col in size_cols_file:
    if col in file_df.columns:
        file_df[col] = pd.to_numeric(file_df[col], errors='coerce').round().astype('Int64')

# Normalise target_df time columns to ms
time_cols_target = ['compile_time_sum_ms', 'compile_time_max_ms', 'total_build_time_ms',
                    'codegen_time_ms', 'archive_time_ms', 'link_time_ms']
for col in time_cols_target:
    if col in target_df.columns:
        target_df[col] = pd.to_numeric(target_df[col], errors='coerce').round().astype('Int64')

print('Type casting complete')

## 8. Write Cleaned Data

In [ ]:
def write_parquet_safe(
    df: pd.DataFrame,
    path: Path,
    schema: pa.Schema | None = None,
    compression: str = 'snappy',
) -> None:
    """Write a DataFrame to parquet, coercing dtypes to match schema where possible."""
    if schema is not None:
        # Only keep columns that are in the schema + any extra we added (like clipped)
        schema_cols = [f.name for f in schema]
        extra_cols  = [c for c in df.columns if c not in schema_cols]
        df_out = df[[c for c in schema_cols if c in df.columns] + extra_cols].copy()
    else:
        df_out = df.copy()
    table = pa.Table.from_pandas(df_out, schema=None, preserve_index=False)
    pq.write_table(table, path, compression=compression)
    print(f'Wrote {len(df_out):,} rows -> {path}  ({path.stat().st_size / 1024:.1f} KB)')

write_parquet_safe(file_df,   cfg.processed_data_dir / 'file_metrics.parquet',   FILE_METRICS_SCHEMA)
write_parquet_safe(target_df, cfg.processed_data_dir / 'target_metrics.parquet', TARGET_METRICS_SCHEMA)
write_parquet_safe(edge_df,   cfg.processed_data_dir / 'edge_list.parquet',      EDGE_LIST_SCHEMA)
write_parquet_safe(contributors_clean, cfg.raw_data_dir / 'contributors_clean.csv'.replace('.csv', '.parquet'))

print('\nAll cleaned parquet files written successfully.')